In [1]:
import pandas as pd
from prepare_data_for_prompts_classifier import detect_english_text
from tqdm import tqdm

C:\Users\vpooz\PycharmProjects\LLM Security Scaner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data=pd.read_json('sft_output.jsonl',lines=True)
#data1=pd.read_json('sft_output_1.jsonl',lines=True)
#data=pd.concat([data,data1])
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10237 entries, 0 to 10236
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          10237 non-null  str   
 1   is_unsafe       10237 non-null  int64 
 2   response        10237 non-null  str   
 3   confidence      10237 non-null  str   
 4   attack_type     10237 non-null  object
 5   explanation     10237 non-null  str   
 6   recommendation  10237 non-null  str   
dtypes: int64(1), object(1), str(5)
memory usage: 13.3+ MB


In [3]:
data=data[data[['prompt','response','explanation']].apply(lambda x :all(detect_english_text(text,min_confidence=0.95)for text in x),axis=1)]

In [4]:
import re
def is_clean_strict(r):
    text = r['explanation'] + ' ' + r['response']

    if re.search(r'[\u0600-\u06ff\u4e00-\u9fff\u3040-\u30ff\uff00-\uffef]', text):
        return False
    return True

data = data[data.apply(is_clean_strict, axis=1)]

In [5]:
data.info()

<class 'pandas.DataFrame'>
Index: 10079 entries, 0 to 10236
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          10079 non-null  str   
 1   is_unsafe       10079 non-null  int64 
 2   response        10079 non-null  str   
 3   confidence      10079 non-null  str   
 4   attack_type     10079 non-null  object
 5   explanation     10079 non-null  str   
 6   recommendation  10079 non-null  str   
dtypes: int64(1), object(1), str(5)
memory usage: 13.1+ MB


In [6]:
data['confidence'].value_counts()

confidence
high      6138
low       3693
medium     227
unsafe      21
Name: count, dtype: int64

In [7]:
data=data[data['confidence'].isin(['high','medium','low'])]
case_1=((data['is_unsafe']==1) & (data['recommendation'].isin(['BLOCK','REVIEW'])))
case_2=((data['is_unsafe']==0) & (data['recommendation'].isin(['SAFE','REVIEW'])))
data=data[case_1|case_2]



In [8]:
data=data.drop_duplicates(subset='prompt')

In [9]:
data.info()

<class 'pandas.DataFrame'>
Index: 9596 entries, 0 to 10236
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          9596 non-null   str   
 1   is_unsafe       9596 non-null   int64 
 2   response        9596 non-null   str   
 3   confidence      9596 non-null   str   
 4   attack_type     9596 non-null   object
 5   explanation     9596 non-null   str   
 6   recommendation  9596 non-null   str   
dtypes: int64(1), object(1), str(5)
memory usage: 12.5+ MB


In [10]:
import pandas as pd
df=data.copy()

print(df['recommendation'].value_counts())
print(df[df['is_unsafe']==1]['recommendation'].value_counts())
print(df[df['is_unsafe']==0]['recommendation'].value_counts())


print("\nПримеры explanation:")
for exp in df['explanation'].sample(5):
    print(exp[:100])

recommendation
SAFE      4618
BLOCK     3464
REVIEW    1514
Name: count, dtype: int64
recommendation
BLOCK     3464
REVIEW    1196
Name: count, dtype: int64
recommendation
SAFE      4618
REVIEW     318
Name: count, dtype: int64

Примеры explanation:
The prompt does not contain any malicious content or intent that would require analysis for security
The prompt injects unsafe instructions and manipulates the model to provide potentially harmful info
The prompt is a request for information on how to start raising chickens for eggs, which is a benign
The website promotes illegal activities which can lead to misinformation and harmful content being s
The prompt does not contain any malicious content or intent and is unrelated to security analysis.


In [11]:
sft_data=pd.read_csv('sft_sampled.csv')
data=pd.merge(data,sft_data[['prompt','from_dataset']],how='left',on='prompt')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 9596 entries, 0 to 9595
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   prompt          9596 non-null   str   
 1   is_unsafe       9596 non-null   int64 
 2   response        9596 non-null   str   
 3   confidence      9596 non-null   str   
 4   attack_type     9596 non-null   object
 5   explanation     9596 non-null   str   
 6   recommendation  9596 non-null   str   
 7   from_dataset    9596 non-null   str   
dtypes: int64(1), object(1), str(6)
memory usage: 12.6+ MB


In [12]:
data.to_json('sft_output_cleaned.jsonl',orient='records',lines=True,force_ascii=False)